# P0 v4 — Thực nghiệm bổ sung: ngưỡng nhãn của BCE (RainS2S)
Chạy 2 ô theo thứ tự (cùng Drive `rainfall_p0`). Ô 1 nạp dữ liệu và định nghĩa mô hình; ô 2 huấn luyện **6 lượt** (2 ngưỡng BCE mới × 3 seed; tự resume nếu bị ngắt, chạy lại ô 2 là tiếp tục), rồi in bảng và tự tải `T7_bce_ablation.zip`. Ước lượng ~1–3 giờ trên A100 (RainS2S 40 epoch/lượt); nên dùng A100.

In [1]:
# ── 0. Cài đặt & mount Drive ─────────────────────────────────────────
import os, sys, json, time, math, random, re, glob, warnings, subprocess, shutil
warnings.filterwarnings('ignore')
try:
    import cv2
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', '-q', 'install', 'opencv-python-headless'], check=True)
    import cv2
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if DEVICE.type == 'cuda':
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.backends.cudnn.benchmark = True
    print('GPU:', torch.cuda.get_device_name(0), '| VRAM %.1f GB' % (torch.cuda.get_device_properties(0).total_memory / 1e9))
else:
    print('⚠️ Không có GPU — chỉ nên dùng cho SMOKE test.')
print('torch', torch.__version__, '| cv2', cv2.__version__)

# ── 1. CẤU HÌNH ──────────────────────────────────────────────────────
SMOKE = bool(int(os.environ.get('SMOKE', '0')))     # SMOKE=1: chạy thử nhỏ để bắt lỗi (không dùng cho paper)

CFG = dict(
    DATA_PATH   = os.environ.get('RAIN_RAW', '/content/drive/MyDrive/rainfall_processed/rain_raw.npy'),  # mm/h, (T,150,80)
    GPM_NC4_DIR = os.environ.get('GPM_DIR',  '/content/drive/MyDrive/gpm_data'),                         # (tùy chọn) *.nc4 để kiểm tra thời gian
    OUT_DIR     = os.environ.get('P0_OUT',   '/content/drive/MyDrive/rainfall_p0'),

    SEEDS       = [0, 1, 2],
    EPOCHS      = 30,      # mô hình một bước
    S2S_EPOCHS  = 40,      # Seq2Seq (teacher forcing giảm 0.9 → 0.1 trong S2S_EPOCHS epoch)
    PATIENCE    = 8,
    BATCH       = 32,
    EVAL_BATCH  = 64,
    LR          = 5e-4,
    MAX_LR      = 2e-3,    # OneCycle
    WD          = 1e-4,
    USE_AMP     = True,    # bf16 khi train; đánh giá luôn fp32/TF32

    SEQ_LEN     = 6,       # 3 giờ lịch sử
    TRAIN_RATIO = 0.70, VAL_RATIO = 0.15,
    # CÙNG một loss cho mọi mô hình (kiểm soát biến). Trọng số w>0 đẩy mô hình về phía "nhạy" (POD cao, bias > 1).
    LOSS        = dict(alpha=0.6, beta=0.3, gamma=0.1, w=3.0),
    LOSS_PLAIN  = dict(alpha=0.7, beta=0.3, gamma=0.0, w=0.0),   # ablation: không nhấn mạnh mưa lớn
    ROLLOUT     = True,    # mô hình một bước: lặp tự hồi quy để có +60/+90 phút
    SAVE_PREDS_SEED = 0,   # lưu dự báo test (fp16) của seed này để vẽ hình định tính
    BOOT_BLOCK  = 48,      # 48 mẫu = 1 ngày
    BOOT_N      = 2000,
    MODELS = ['cnn_lstm', 'cnn_bilstm', 'cnn_gru', 'convlstm_noatt', 'convlstm_att',
              'unet_axial', 'unet_axial_plainloss', 'seq2seq_unet',
              'simvp_gsta', 'predrnnpp'],          # v2: hai baseline hiện đại (dự báo thẳng 3 khung)
    # siêu tham số riêng theo mô hình (theo cấu hình gốc): PredRNN++ dùng lr thấp
    HP = {'predrnnpp': dict(LR=1e-4, MAX_LR=5e-4)},
)
if os.environ.get('MODELS'): CFG['MODELS'] = os.environ['MODELS'].split(',')   # ví dụ MODELS=simvp_gsta
THR = [0.1, 0.5, 1.0, 2.0, 5.0, 10.0]      # ngưỡng mưa (mm/h) cho đánh giá đa ngưỡng
FSS_SCALES = [1, 3, 5, 9, 17]              # pixel; 1 pixel = 0.1°
EVENT_THR = 0.5

if SMOKE:
    CFG.update(SEEDS=[0], EPOCHS=1, S2S_EPOCHS=1, BATCH=8, EVAL_BATCH=16, USE_AMP=False, BOOT_N=100,
               OUT_DIR=os.environ.get('P0_OUT', 'smoke_out'))
if os.environ.get('S2S_EPOCHS'): CFG['S2S_EPOCHS'] = int(os.environ['S2S_EPOCHS'])
os.makedirs(CFG['OUT_DIR'], exist_ok=True)
for d in ('ckpt', 'results', 'preds', 'tables', 'figs'):
    os.makedirs(os.path.join(CFG['OUT_DIR'], d), exist_ok=True)
OUT = CFG['OUT_DIR']
print('OUT_DIR:', OUT, '| SMOKE' if SMOKE else '')

# ── 2. DỮ LIỆU & GIAO THỨC ───────────────────────────────────────────
raw_all = np.load(CFG['DATA_PATH'], mmap_mode='r')
raw = np.ascontiguousarray(raw_all[:360, :64, :40] if SMOKE else raw_all).astype(np.float32)
assert raw.min() >= 0, 'rain_raw phải là mm/h đã thay giá trị âm/NaN bằng 0'
T, H, W = raw.shape
ntr, nva = int(T * CFG['TRAIN_RATIO']), int(T * CFG['VAL_RATIO'])
SEG = dict(train=(0, ntr), val=(ntr, ntr + nva), test=(ntr + nva, T))

# Chuẩn hóa log1p / LOG_MAX với LOG_MAX tính CHỈ trên train (sửa rò rỉ chuẩn hóa của NB1)
LOG_MAX = float(np.log1p(raw[:ntr]).max())
def norm_np(x): return np.log1p(x) / LOG_MAX
def den(x):     return torch.expm1(x * LOG_MAX)

Z = {k: torch.from_numpy(norm_np(raw[a:b])).float().to(DEVICE) for k, (a, b) in SEG.items()}   # chuẩn hóa
R = {k: torch.from_numpy(raw[a:b]).float().to(DEVICE) for k, (a, b) in SEG.items() if k != 'train'}  # mm/h gốc (đích đánh giá)
NS = {k: len(Z[k]) - CFG['SEQ_LEN'] - 3 + 1 for k in ('val', 'test')}   # số mẫu có đủ 3 thời hạn (+30/+60/+90)
AR6 = torch.arange(CFG['SEQ_LEN'], device=DEVICE)

stats = dict(T=T, H=H, W=W, n_train=ntr, n_val=nva, n_test=T - ntr - nva, n_eval_samples=NS,
             log_max_train=LOG_MAX, raw_max_all=float(raw.max()), raw_max_train=float(raw[:ntr].max()),
             frac_zero=float((raw == 0).mean()), frac_gt_0p1=float((raw > 0.1).mean()), frac_gt_10=float((raw > 10).mean()),
             test_pixels_above_train_max=float((raw[ntr + nva:] > raw[:ntr].max()).mean()),
             frac_rain_gt_0p5={k: float((raw[a:b] > 0.5).mean()) for k, (a, b) in SEG.items()})
json.dump(stats, open(f'{OUT}/results/data_stats.json', 'w'), indent=1)
print(json.dumps(stats, indent=1))

# ── 3. (TÙY CHỌN) KIỂM TRA LIÊN TỤC THỜI GIAN từ tên file .nc4 ────────
# Trả lời: các khung có thực sự cách nhau 30 phút không? Có khoảng trống nào bị cửa sổ trượt "nối" qua không?
TIMES = None
pat = re.compile(r'IMERG\.(\d{8})-S(\d{6})')
files = sorted(glob.glob(os.path.join(CFG['GPM_NC4_DIR'], '*.nc4')))
if files:
    import datetime as dt
    ts = []
    for f in files:
        m = pat.search(os.path.basename(f))
        if m: ts.append(dt.datetime.strptime(m.group(1) + m.group(2), '%Y%m%d%H%M%S'))
    print(f'{len(files)} file .nc4, {len(ts)} có timestamp trong tên')
    if len(ts) == len(files):
        d = np.diff(np.array(ts, dtype='datetime64[s]')).astype('int64') / 60
        gaps = np.where(d != 30)[0]
        print('Khoảng thời gian:', ts[0], '→', ts[-1], '| số bước kỳ vọng nếu liên tục:', int((ts[-1] - ts[0]).total_seconds() / 1800) + 1)
        print('Số khoảng trống (bước ≠ 30 phút):', len(gaps))
        for g in gaps[:15]: print('   ', ts[g], '→', ts[g + 1], f'({d[g]:.0f} phút)')
        if len(ts) == T:
            TIMES = ts
            a, b = SEG['test']; print('Tập test:', TIMES[a], '→', TIMES[b - 1])
            json.dump(dict(first=str(ts[0]), last=str(ts[-1]), n_gaps=int(len(gaps)), test_start=str(TIMES[a]), test_end=str(TIMES[b - 1])),
                      open(f'{OUT}/results/time_audit.json', 'w'), indent=1)
        else:
            print(f'⚠️ Số file ({len(ts)}) ≠ T ({T}): không gán được timestamp cho từng khung.')
else:
    print('Không thấy .nc4 → bỏ qua kiểm tra thời gian. (Ghi vào paper: "chưa kiểm tra tính liên tục 30 phút".)')
    print('Gợi ý: danh sách tải gốc có 35.040 file (2022-01-01 → 2023-12-31) nhưng dữ liệu có T =', T, '→ nên kiểm tra có file bị thiếu không.')

# ── 4. THƯ VIỆN ĐÁNH GIÁ (GPU) ───────────────────────────────────────
class Acc:
    # cộng dồn bảng nhầm lẫn theo từng ngưỡng + FSS + sai số liên tục; lưu đếm theo từng mẫu cho ngưỡng sự kiện (phục vụ bootstrap)
    def __init__(self):
        self.c = torch.zeros(len(THR), 4, dtype=torch.float64)          # H, M, F, CN
        self.se = 0.0; self.ae = 0.0; self.n = 0
        self.fss = {s: torch.zeros(2, dtype=torch.float64) for s in FSS_SCALES}
        self.per = []
    @torch.no_grad()
    def add(self, p, o):                                                # p, o: (b,H,W) mm/h, fp32
        d = (p - o).double(); self.se += float((d ** 2).sum()); self.ae += float(d.abs().sum()); self.n += o.numel()
        tot = o[0].numel()
        for j, t in enumerate(THR):
            pb, ob = p > t, o > t
            h = (pb & ob).flatten(1).sum(1); m = (~pb & ob).flatten(1).sum(1); f = (pb & ~ob).flatten(1).sum(1)
            self.c[j] += torch.stack([h.sum(), m.sum(), f.sum(), (tot * len(o) - h.sum() - m.sum() - f.sum())]).double().cpu()
            if t == EVENT_THR:
                self.per.append(torch.stack([h, m, f, tot - h - m - f], 1).cpu().numpy())
        pb = (p > EVENT_THR).float().unsqueeze(1); ob = (o > EVENT_THR).float().unsqueeze(1)
        for s in FSS_SCALES:
            fp = F.avg_pool2d(pb, s, 1, s // 2); fo = F.avg_pool2d(ob, s, 1, s // 2)
            self.fss[s] += torch.stack([((fp - fo) ** 2).sum(), (fp ** 2 + fo ** 2).sum()]).double().cpu()
    def counts(self): return np.concatenate(self.per, 0)               # (n,4)
    def res(self, thr=EVENT_THR, with_rmse=True):
        H_, M_, F_, CN = self.c[THR.index(thr)].tolist(); n = H_ + M_ + F_ + CN; e = 1e-12
        hr = (H_ + M_) * (H_ + F_) / n
        r = dict(CSI=H_ / (H_ + M_ + F_ + e), POD=H_ / (H_ + M_ + e), FAR=F_ / (H_ + F_ + e), Bias=(H_ + F_) / (H_ + M_ + e),
                 ETS=(H_ - hr) / (H_ + M_ + F_ - hr + e),
                 HSS=2 * (H_ * CN - F_ * M_) / ((H_ + M_) * (M_ + CN) + (H_ + F_) * (F_ + CN) + e))
        if with_rmse: r.update(RMSE=(self.se / self.n) ** .5, MAE=self.ae / self.n)
        if thr == EVENT_THR: r.update({f'FSS{s}': 1 - self.fss[s][0].item() / (self.fss[s][1].item() + e) for s in FSS_SCALES})
        return r

def csi_pod_far_bias(c):                                                # c: (...,4) numpy
    H_, M_, F_, _ = [c[..., i] for i in range(4)]; e = 1e-12
    return H_ / (H_ + M_ + F_ + e), H_ / (H_ + M_ + e), F_ / (H_ + F_ + e), (H_ + F_) / (H_ + M_ + e)

def obs_of(split, h, i0, i1): return R[split][i0 + CFG['SEQ_LEN'] - 1 + h: i1 + CFG['SEQ_LEN'] - 1 + h]
print('metrics ready')

# ── 5. BASELINE: persistence + dòng quang học ────────────────────────
def save_json(path, obj): json.dump(obj, open(path, 'w'), indent=1, default=lambda o: float(o))

def eval_fixed(name, fn, chunk=128):
    # fn(i0,i1) -> {h: (b,H,W) mm/h tensor}; đánh giá trên tập test với 3 thời hạn
    n = NS['test']; acc = {h: Acc() for h in (1, 2, 3)}
    for i0 in range(0, n, chunk):
        i1 = min(i0 + chunk, n); P = fn(i0, i1)
        for h in (1, 2, 3): acc[h].add(P[h], obs_of('test', h, i0, i1))
    out = dict(name=name, seed=None, params=0, h={}); cnt = {}
    for h in (1, 2, 3):
        out['h'][str(h)] = dict(raw=acc[h].res(), thr={str(t): acc[h].res(t) for t in THR})
        cnt[f'h{h}_raw'] = acc[h].counts()
    save_json(f'{OUT}/results/{name}.json', out); np.savez_compressed(f'{OUT}/results/{name}_counts.npz', **cnt)
    return out

def persistence_fn(i0, i1):
    last = R['test'][i0 + CFG['SEQ_LEN'] - 1: i1 + CFG['SEQ_LEN'] - 1]
    return {h: last for h in (1, 2, 3)}

_gy, _gx = np.mgrid[0:H, 0:W].astype(np.float32)
_raw_test = raw[SEG['test'][0]:SEG['test'][1]]
def optflow_fn(i0, i1):
    outs = {h: np.zeros((i1 - i0, H, W), np.float32) for h in (1, 2, 3)}
    for k, i in enumerate(range(i0, i1)):
        pv, cu = _raw_test[i + CFG['SEQ_LEN'] - 2], _raw_test[i + CFG['SEQ_LEN'] - 1]
        im = lambda x: np.clip(np.log1p(x) / LOG_MAX * 255, 0, 255).astype(np.uint8)
        f = cv2.calcOpticalFlowFarneback(im(pv), im(cu), None, 0.5, 3, 15, 3, 5, 1.2, 0)
        for h in (1, 2, 3):    # advection hằng số: dịch trường mưa cuối theo vector dòng * h bước
            outs[h][k] = cv2.remap(cu, _gx - h * f[..., 0], _gy - h * f[..., 1], cv2.INTER_LINEAR, borderMode=cv2.BORDER_CONSTANT, borderValue=0)
    return {h: torch.from_numpy(v).to(DEVICE) for h, v in outs.items()}

BASE = {}
for nm, fn in [('persistence', persistence_fn), ('optflow', optflow_fn)]:
    t0 = time.time(); BASE[nm] = eval_fixed(nm, fn)
    print(nm, f'({time.time() - t0:.0f}s)', {h: {k: round(BASE[nm]['h'][h]['raw'][k], 4) for k in ('RMSE', 'CSI', 'POD', 'FAR', 'Bias')} for h in BASE[nm]['h']})
print('\nĐối chiếu bài cũ (persistence, 4173 mẫu test): +30 CSI 0.6554 POD 0.7918 FAR 0.2081 | +60 CSI 0.5192 | +90 CSI 0.4273  → phải khớp nếu dùng đúng dữ liệu & cách chia.')

# ── 6. MÔ HÌNH ───────────────────────────────────────────────────────
# 6a. CNN-RNN dạng vector (NB4): CNN → (Bi)LSTM/GRU → Temporal Attention → CNN decoder
class SpatialCNNEncoder(nn.Module):
    def __init__(s, feat=64):
        super().__init__()
        s.cnn = nn.Sequential(nn.Conv2d(1, 32, 3, 2, 1), nn.BatchNorm2d(32), nn.ReLU(True),
                              nn.Conv2d(32, feat, 3, 2, 1), nn.BatchNorm2d(feat), nn.ReLU(True),
                              nn.Conv2d(feat, feat, 3, 1, 1), nn.BatchNorm2d(feat), nn.ReLU(True))
    def forward(s, x):                                                  # (B,T,1,H,W) → (B,T,feat)
        B, T = x.shape[:2]; y = s.cnn(x.flatten(0, 1)).mean((-1, -2)); return y.view(B, T, -1)

class TemporalAttention(nn.Module):
    def __init__(s, d):
        super().__init__(); s.W = nn.Linear(d, d, bias=False); s.v = nn.Linear(d, 1, bias=False)
    def forward(s, x):
        w = F.softmax(s.v(torch.tanh(s.W(x))).squeeze(-1), 1); return (w.unsqueeze(-1) * x).sum(1), w

class SpatialCNNDecoder(nn.Module):
    def __init__(s, d, H, W):
        super().__init__(); s.H, s.W, s.h4, s.w4 = H, W, H // 4, W // 4
        s.proj = nn.Linear(d, 64 * s.h4 * s.w4)
        s.up = nn.Sequential(nn.ConvTranspose2d(64, 32, 4, 2, 1), nn.BatchNorm2d(32), nn.ReLU(True),
                             nn.ConvTranspose2d(32, 16, 4, 2, 1), nn.BatchNorm2d(16), nn.ReLU(True),
                             nn.Conv2d(16, 1, 3, padding=1), nn.Sigmoid())
    def forward(s, x):
        y = s.up(s.proj(x).view(x.size(0), 64, s.h4, s.w4))
        if y.shape[-2:] != (s.H, s.W): y = F.interpolate(y, size=(s.H, s.W), mode='bilinear', align_corners=False)
        return y.squeeze(1)

class CNNRNN(nn.Module):
    def __init__(s, rnn='lstm', feat=64, hid=128, layers=2, drop=0.3):
        super().__init__(); s.enc = SpatialCNNEncoder(feat)
        bi = rnn == 'bilstm'; cls = nn.GRU if rnn == 'gru' else nn.LSTM
        s.rnn = cls(feat, hid, layers, batch_first=True, dropout=drop, bidirectional=bi)
        d = hid * (2 if bi else 1); s.att = TemporalAttention(d); s.dec = SpatialCNNDecoder(d, H, W); s.drop = nn.Dropout(drop)
    def forward(s, x):
        o, _ = s.rnn(s.enc(x)); ctx, _ = s.att(s.drop(o)); return s.dec(ctx)

# 6b. ConvLSTM (NB2): Conv3D encoder → ConvLSTM 2 lớp → (spatial+temporal attention) → decoder
class ConvLSTMCell(nn.Module):
    def __init__(s, i, h, ks=3):
        super().__init__(); s.hid = h; s.conv = nn.Conv2d(i + h, 4 * h, ks, padding=ks // 2)
    def forward(s, x, h, c):
        i, f, g, o = s.conv(torch.cat([x, h], 1)).chunk(4, 1)
        c = torch.sigmoid(f) * c + torch.sigmoid(i) * torch.tanh(g); return torch.sigmoid(o) * torch.tanh(c), c

class ConvLSTMBlock(nn.Module):                                         # (B,T,C,H,W) → all_h, last_h, last_c
    def __init__(s, i, h, ks=3):
        super().__init__(); s.cell = ConvLSTMCell(i, h, ks); s.hid = h
    def forward(s, x, h=None, c=None):
        B, T, _, Hh, Ww = x.shape
        h = torch.zeros(B, s.hid, Hh, Ww, device=x.device, dtype=x.dtype) if h is None else h
        c = torch.zeros_like(h) if c is None else c
        hs = []
        for t in range(T):
            h, c = s.cell(x[:, t], h, c); hs.append(h)
        return torch.stack(hs, 1), h, c

class ConvLSTMStack(nn.Module):                                         # nhiều lớp ConvLSTM, trả toàn bộ chuỗi
    def __init__(s, i, hids):
        super().__init__(); d = [i] + list(hids); s.cells = nn.ModuleList([ConvLSTMCell(d[k], d[k + 1]) for k in range(len(hids))])
    def forward(s, x):
        B, T, _, Hh, Ww = x.shape
        st = [(torch.zeros(B, c.hid, Hh, Ww, device=x.device, dtype=x.dtype),) * 2 for c in s.cells]
        outs = []
        for t in range(T):
            xt = x[:, t]
            for l, cell in enumerate(s.cells):
                h, c = cell(xt, *st[l]); st[l] = (h, c); xt = h
            outs.append(xt)
        return torch.stack(outs, 1)

class ConvLSTMNet(nn.Module):
    def __init__(s, att=True, enc_out=32, hid=(64, 64)):
        super().__init__(); s.att = att; ch = hid[-1]
        s.enc = nn.Sequential(nn.Conv3d(1, 16, 3, padding=1), nn.BatchNorm3d(16), nn.ReLU(True),
                              nn.Conv3d(16, enc_out, 3, padding=1), nn.BatchNorm3d(enc_out), nn.ReLU(True))
        s.lstm = ConvLSTMStack(enc_out, hid)
        if att:
            s.sp = nn.Sequential(nn.Conv2d(ch, ch // 4, 3, padding=1), nn.ReLU(True), nn.Conv2d(ch // 4, 1, 1))
            s.te = nn.Sequential(nn.Conv1d(ch, ch // 2, 3, padding=1), nn.ReLU(True), nn.Conv1d(ch // 2, 1, 1))
        s.dec = nn.Sequential(nn.Conv2d(ch, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(True),
                              nn.Conv2d(32, 16, 3, padding=1), nn.ReLU(True), nn.Conv2d(16, 1, 1), nn.Sigmoid())
    def forward(s, x):                                                  # (B,T,1,H,W)
        f = s.lstm(s.enc(x.permute(0, 2, 1, 3, 4)).permute(0, 2, 1, 3, 4))
        if s.att:
            B, T, C, Hh, Ww = f.shape
            f = f * torch.sigmoid(s.sp(f.flatten(0, 1))).view(B, T, 1, Hh, Ww)                    # spatial attention
            w = F.softmax(s.te(f.mean((-1, -2)).permute(0, 2, 1)), 2).permute(0, 2, 1)[..., None, None]  # temporal attention
            last = (f * w).sum(1)
        else:
            last = f[:, -1]
        return s.dec(last).squeeze(1)

# 6c. ConvLSTM-UNet + Axial Attention (NB5) và Seq2Seq (NB6)
class EncBlock(nn.Module):
    def __init__(s, i, o):
        super().__init__(); s.lstm = ConvLSTMBlock(i, o); s.bn = nn.BatchNorm2d(o)
    def forward(s, x):
        a, h, c = s.lstm(x); return a, F.relu(s.bn(h)), c

class AxialAttention(nn.Module):
    def __init__(s, ch, heads=4):
        super().__init__(); assert ch % heads == 0
        s.h, s.d = heads, ch // heads; s.scale = s.d ** -0.5
        s.qh = nn.Conv1d(ch, 3 * ch, 1, bias=False); s.qw = nn.Conv1d(ch, 3 * ch, 1, bias=False)
        s.proj = nn.Conv2d(ch, ch, 1); s.bn = nn.BatchNorm2d(ch)
    def _ax(s, x, qkv):                                                 # x: (BN,C,L)
        BN, C, L = x.shape; q, k, v = qkv(x).chunk(3, 1)
        q, k, v = [t.view(BN, s.h, s.d, L).permute(0, 1, 3, 2) for t in (q, k, v)]
        a = F.softmax((q @ k.transpose(-2, -1)) * s.scale, -1)
        return (a @ v).permute(0, 1, 3, 2).reshape(BN, C, L)
    def forward(s, x):
        B, C, Hh, Ww = x.shape
        xh = s._ax(x.permute(0, 3, 1, 2).reshape(B * Ww, C, Hh), s.qh).view(B, Ww, C, Hh).permute(0, 2, 3, 1)
        xw = s._ax(xh.permute(0, 2, 1, 3).reshape(B * Hh, C, Ww), s.qw).view(B, Hh, C, Ww).permute(0, 2, 1, 3)
        return F.relu(s.bn(s.proj(xw) + x))

class UDec(nn.Module):
    def __init__(s, i, sk, o):
        super().__init__(); s.up = nn.ConvTranspose2d(i, i, 2, 2)
        s.conv = nn.Sequential(nn.Conv2d(i + sk, o, 3, padding=1), nn.BatchNorm2d(o), nn.ReLU(True),
                               nn.Conv2d(o, o, 3, padding=1), nn.BatchNorm2d(o), nn.ReLU(True))
    def forward(s, x, skip):
        x = s.up(x)
        if x.shape[-2:] != skip.shape[-2:]: x = F.interpolate(x, size=skip.shape[-2:], mode='bilinear', align_corners=False)
        return s.conv(torch.cat([x, skip], 1))

class ConvLSTMUNet(nn.Module):
    def __init__(s, c1=32, c2=64, c3=128, cb=128, heads=4, pred_len=1):
        super().__init__(); s.P = pred_len
        s.e1, s.e2, s.e3, s.e4 = EncBlock(1, c1), EncBlock(c1, c2), EncBlock(c2, c3), EncBlock(c3, cb)
        s.pool = nn.MaxPool2d(2, 2); s.axial = AxialAttention(cb, heads); s.drop = nn.Dropout2d(0.2)
        s.d3, s.d2, s.d1 = UDec(cb, c3, c3 // 2), UDec(c3 // 2, c2, c2 // 2), UDec(c2 // 2, c1, c1)
        s.head = nn.Sequential(nn.Conv2d(c1, c1 // 2, 3, padding=1), nn.BatchNorm2d(c1 // 2), nn.ReLU(True), nn.Conv2d(c1 // 2, 1, 1), nn.Sigmoid())
        if pred_len > 1:                                                # Seq2Seq: decoder ConvLSTM tự hồi quy ở độ phân giải bottleneck
            s.dec_cell = ConvLSTMCell(1 + cb, cb); s.dec_proj = nn.Sequential(nn.Conv2d(cb, cb, 3, padding=1), nn.BatchNorm2d(cb), nn.ReLU(True))
    def _pool(s, x):
        B, T, C, Hh, Ww = x.shape; y = s.pool(x.flatten(0, 1)); return y.view(B, T, C, *y.shape[-2:])
    def _enc(s, x):
        a1, s1, _ = s.e1(x); a2, s2, _ = s.e2(s._pool(a1)); a3, s3, _ = s.e3(s._pool(a2)); _, bn, bc = s.e4(s._pool(a3))
        return s1, s2, s3, s.axial(s.drop(bn)), bc
    def _dec(s, feat, s1, s2, s3):
        o = s.head(s.d1(s.d2(s.d3(feat, s3), s2), s1)).squeeze(1)
        return o if o.shape[-2:] == (H, W) else F.interpolate(o.unsqueeze(1), size=(H, W), mode='bilinear', align_corners=False).squeeze(1)
    def forward(s, x, target=None, tf=0.0):                             # x: (B,T,1,H,W); target: (B,P,H,W)
        s1, s2, s3, ctx, bc = s._enc(x)
        if s.P == 1: return s._dec(ctx, s1, s2, s3)
        h, c = ctx, bc; prev = F.adaptive_avg_pool2d(x[:, -1], ctx.shape[-2:]); preds = []
        for t in range(s.P):
            h, c = s.dec_cell(torch.cat([prev, ctx], 1), h, c)
            p = s._dec(s.dec_proj(h), s1, s2, s3); preds.append(p)
            use_gt = target is not None and tf > 0 and t < s.P - 1 and torch.rand(1).item() < tf
            prev = F.adaptive_avg_pool2d((target[:, t] if use_gt else p.detach()).unsqueeze(1), ctx.shape[-2:])
        return torch.stack(preds, 1)

# 6d. SimVPv2 (gSTA) — theo OpenSTL: encoder/decoder ConvSC + translator gồm các khối Gated Spatiotemporal Attention.
#     Điều chỉnh cho bài này: đệm H,W lên bội của 4 rồi cắt lại; sigmoid ở đầu ra (loss chung cần [0,1]); lấy 3 khung đầu.
def drop_path(x, p, training):
    if p == 0.0 or not training: return x
    keep = 1 - p; m = x.new_empty(x.shape[0], 1, 1, 1).bernoulli_(keep); return x * m / keep

class ConvSC(nn.Module):
    def __init__(s, ci, co, k=3, down=False, up=False):
        super().__init__(); st = 2 if down else 1; pad = (k - st + 1) // 2; s.up = up
        s.conv = nn.Conv2d(ci, co * 4, k, 1, pad) if up else nn.Conv2d(ci, co, k, st, pad)
        s.ps = nn.PixelShuffle(2) if up else nn.Identity(); s.norm = nn.GroupNorm(2, co); s.act = nn.SiLU()
    def forward(s, x): return s.act(s.norm(s.ps(s.conv(x))))

class GABlock(nn.Module):
    def __init__(s, dim, kernel=21, dil=3, mlp_ratio=4.0, dp=0.0):
        super().__init__(); dk = 2 * dil - 1; ddk = kernel // dil + ((kernel // dil) % 2 - 1); hid = int(dim * mlp_ratio)
        s.norm1 = nn.BatchNorm2d(dim); s.proj1 = nn.Conv2d(dim, dim, 1); s.act = nn.GELU()
        s.conv0 = nn.Conv2d(dim, dim, dk, padding=(dk - 1) // 2, groups=dim)
        s.conv_sp = nn.Conv2d(dim, dim, ddk, padding=dil * (ddk - 1) // 2, groups=dim, dilation=dil)
        s.conv1 = nn.Conv2d(dim, 2 * dim, 1); s.proj2 = nn.Conv2d(dim, dim, 1)
        s.norm2 = nn.BatchNorm2d(dim); s.fc1 = nn.Conv2d(dim, hid, 1); s.dw = nn.Conv2d(hid, hid, 3, 1, 1, groups=hid); s.fc2 = nn.Conv2d(hid, dim, 1)
        s.ls1 = nn.Parameter(1e-2 * torch.ones(dim)); s.ls2 = nn.Parameter(1e-2 * torch.ones(dim)); s.dp = dp
        for m in s.modules():
            if isinstance(m, nn.Conv2d):
                fan = m.kernel_size[0] * m.kernel_size[1] * m.out_channels // m.groups
                nn.init.normal_(m.weight, 0, math.sqrt(2.0 / fan)); nn.init.zeros_(m.bias)
    def attn(s, x):
        sc = x; x = s.act(s.proj1(x)); f, g = s.conv1(s.conv_sp(s.conv0(x))).chunk(2, 1)
        return s.proj2(torch.sigmoid(g) * f) + sc
    def forward(s, x):
        x = x + drop_path(s.ls1[None, :, None, None] * s.attn(s.norm1(x)), s.dp, s.training)
        return x + drop_path(s.ls2[None, :, None, None] * s.fc2(s.act(s.dw(s.fc1(s.norm2(x))))), s.dp, s.training)

class SimVP(nn.Module):
    def __init__(s, T=6, P=3, hid_S=32, hid_T=256, N_S=4, N_T=8, k=3, dp=0.1):
        super().__init__(); s.T, s.P = T, P; samp = [False, True] * (N_S // 2)
        s.enc = nn.ModuleList([ConvSC(1, hid_S, k, down=samp[0])] + [ConvSC(hid_S, hid_S, k, down=d) for d in samp[1:]])
        rs = samp[::-1]; s.dec = nn.ModuleList([ConvSC(hid_S, hid_S, k, up=u) for u in rs[:-1]] + [ConvSC(hid_S, hid_S, k, up=rs[-1])])
        s.readout = nn.Conv2d(hid_S, 1, 1); cin = T * hid_S; dpr = torch.linspace(1e-2, dp, N_T).tolist()
        dims = [cin] + [hid_T] * (N_T - 1) + [cin]; s.mid = nn.ModuleList(); s.red = nn.ModuleList()
        for i in range(N_T):
            s.mid.append(GABlock(dims[i], dp=dpr[i])); s.red.append(nn.Conv2d(dims[i], dims[i + 1], 1) if dims[i] != dims[i + 1] else nn.Identity())
    def forward(s, x, target=None, tf=0.0):                              # x: (B,T,1,H,W) → (B,P,H,W)
        B, T, _, Hh, Ww = x.shape; ph, pw = (-Hh) % 4, (-Ww) % 4
        z = F.pad(x.flatten(0, 1), (0, pw, 0, ph)); e1 = s.enc[0](z); z = e1
        for l in s.enc[1:]: z = l(z)
        _, C_, H_, W_ = z.shape; z = z.view(B, T * C_, H_, W_)
        for blk, red in zip(s.mid, s.red): z = red(blk(z))
        z = z.reshape(B * T, C_, H_, W_)
        for l in s.dec[:-1]: z = l(z)
        y = torch.sigmoid(s.readout(s.dec[-1](z + e1))).view(B, T, Hh + ph, Ww + pw)
        return y[:, :s.P, :Hh, :Ww]

# 6e. PredRNN++ — Causal LSTM + Gradient Highway Unit (Wang et al., ICML 2018), theo OpenSTL.
#     Điều chỉnh: gộp ô 4×4 như cấu hình gốc (đệm 150→152; lưới 38×20, 16 kênh); 4 lớp × 64 kênh; sigmoid ở đầu ra;
#     scheduled sampling dùng cùng lịch teacher forcing với Seq2Seq; giám sát cả các khung đầu vào (như bản gốc) qua model.aux.
class CausalLSTMCell(nn.Module):
    def __init__(s, ci, nh, k=5):
        super().__init__(); s.nh = nh; p = k // 2; c = lambda a, b, kk=k, pp=p: nn.Conv2d(a, b, kk, 1, pp, bias=False)
        s.cx, s.ch, s.cc, s.cm = c(ci, nh * 7), c(nh, nh * 4), c(nh, nh * 3), c(nh, nh * 3)
        s.c2m, s.com, s.last = c(nh, nh * 4), c(nh, nh), nn.Conv2d(nh * 2, nh, 1, bias=False)
    def forward(s, x, h, c, m):
        ix, fx, gx, ixp, fxp, gxp, ox = s.cx(x).chunk(7, 1); ih, fh, gh, oh = s.ch(h).chunk(4, 1)
        im, fm, mm = s.cm(m).chunk(3, 1); ic, fc, gc = s.cc(c).chunk(3, 1)
        c_new = torch.sigmoid(fx + fh + fc + 1.0) * c + torch.sigmoid(ix + ih + ic) * torch.tanh(gx + gh + gc)
        ic2, gc2, fc2, oc2 = s.c2m(c_new).chunk(4, 1)
        m_new = torch.sigmoid(fxp + fm + fc2 + 1.0) * torch.tanh(mm) + torch.sigmoid(ixp + im + ic2) * torch.tanh(gxp + gc2)
        o = torch.tanh(ox + oh + oc2 + s.com(m_new))
        return o * torch.tanh(s.last(torch.cat([c_new, m_new], 1))), c_new, m_new

class GHU(nn.Module):
    def __init__(s, nh, k=5):
        super().__init__(); s.cx = nn.Conv2d(nh, 2 * nh, k, 1, k // 2, bias=False); s.cz = nn.Conv2d(nh, 2 * nh, k, 1, k // 2, bias=False)
    def forward(s, x, z):
        p, u = (s.cx(x) + s.cz(z)).chunk(2, 1); u = torch.sigmoid(u); return u * torch.tanh(p) + (1 - u) * z

class PredRNNpp(nn.Module):
    def __init__(s, nh=64, layers=4, k=5, patch=4, P=3):
        super().__init__(); s.p, s.P, s.nh, s.L = patch, P, nh, layers; C = patch * patch
        s.cells = nn.ModuleList([CausalLSTMCell(C if i == 0 else nh, nh, k) for i in range(layers)])
        s.ghu = GHU(nh, k); s.out = nn.Conv2d(nh, C, 1, bias=False); s.aux = None
    def _pat(s, x):                                                     # (B,T,H,W) → (B,T,p²,H/p,W/p)
        B, T, Hh, Ww = x.shape; p = s.p
        return x.view(B, T, Hh // p, p, Ww // p, p).permute(0, 1, 3, 5, 2, 4).reshape(B, T, p * p, Hh // p, Ww // p)
    def _unpat(s, y):                                                   # (B,T,p²,h,w) → (B,T,H,W)
        B, T, _, h, w = y.shape; p = s.p
        return y.view(B, T, p, p, h, w).permute(0, 1, 4, 2, 5, 3).reshape(B, T, h * p, w * p)
    def forward(s, x, target=None, tf=0.0):                              # x: (B,T,1,H,W); target: (B,P,H,W)
        fr0 = x.squeeze(2); B, T, Hh, Ww = fr0.shape; pd = (0, (-Ww) % s.p, 0, (-Hh) % s.p)   # đệm lên bội của patch
        xin = s._pat(F.pad(fr0, pd)); tg = s._pat(F.pad(target, pd)) if target is not None else None
        h_, w_ = xin.shape[-2:]; z0 = lambda: torch.zeros(B, s.nh, h_, w_, device=x.device, dtype=x.dtype)
        h = [z0() for _ in range(s.L)]; c = [z0() for _ in range(s.L)]; m = z0(); zt = z0(); outs = []; gen = None
        for t in range(T + s.P - 1):
            if t < T: net = xin[:, t]
            elif tg is not None and tf > 0:                             # scheduled sampling theo từng mẫu
                mk = (torch.rand(B, 1, 1, 1, device=x.device) < tf).to(gen.dtype); net = mk * tg[:, t - T].to(gen.dtype) + (1 - mk) * gen
            else: net = gen
            h[0], c[0], m = s.cells[0](net, h[0], c[0], m); zt = s.ghu(h[0], zt)
            h[1], c[1], m = s.cells[1](zt, h[1], c[1], m)
            for i in range(2, s.L): h[i], c[i], m = s.cells[i](h[i - 1], h[i], c[i], m)
            gen = torch.sigmoid(s.out(h[-1])); outs.append(gen)
        y = s._unpat(torch.stack(outs, 1))[..., :Hh, :Ww]                 # y[:, j] dự báo khung j+1
        s.aux = (y[:, :T - 1], fr0[:, 1:]) if (s.training and target is not None) else None
        return y[:, T - 1:]

REG = {   # tên → (hàm tạo, loại, loss)
    'cnn_lstm':              (lambda: CNNRNN('lstm'),   'one', 'LOSS'),
    'cnn_bilstm':            (lambda: CNNRNN('bilstm'), 'one', 'LOSS'),
    'cnn_gru':               (lambda: CNNRNN('gru'),    'one', 'LOSS'),
    'convlstm_noatt':        (lambda: ConvLSTMNet(att=False), 'one', 'LOSS'),
    'convlstm_att':          (lambda: ConvLSTMNet(att=True),  'one', 'LOSS'),
    'unet_axial':            (lambda: ConvLSTMUNet(),   'one', 'LOSS'),
    'unet_axial_plainloss':  (lambda: ConvLSTMUNet(),   'one', 'LOSS_PLAIN'),
    'seq2seq_unet':          (lambda: ConvLSTMUNet(pred_len=3), 's2s', 'LOSS'),
    'simvp_gsta':            (lambda: SimVP(T=CFG['SEQ_LEN'], P=3), 's2s', 'LOSS'),
    'predrnnpp':             (lambda: PredRNNpp(P=3), 's2s', 'LOSS'),
}
NPARAM = {}
for k, (mk, kind, _) in REG.items():
    m = mk(); NPARAM[k] = sum(p.numel() for p in m.parameters() if p.requires_grad)
    with torch.no_grad():
        o = m.eval()(torch.zeros(2, CFG['SEQ_LEN'], 1, H, W)); assert o.shape[-2:] == (H, W), (k, o.shape)
    print(f'{k:22s} {NPARAM[k]:>11,d} tham số  | out {tuple(o.shape)}')
del m, o

# ── 7. LOSS, BATCH, HUẤN LUYỆN ───────────────────────────────────────
class RainLoss(nn.Module):
    # alpha·WMSE + beta·BCE(ngưỡng 0.05 chuẩn hóa) + gamma·TV ; với Seq2Seq: trọng số bước giảm dần (decay^k)
    def __init__(s, alpha, beta, gamma, w, thr=0.05, P=1, decay=0.85):
        super().__init__(); s.a, s.b, s.g, s.w, s.thr = alpha, beta, gamma, w, thr
        sw = [decay ** i for i in range(P)]; s.sw = [x / sum(sw) for x in sw]
    def step(s, p, t):
        wmse = ((1 + s.w * t) * (p - t) ** 2).mean()
        bce = F.binary_cross_entropy(p.clamp(1e-6, 1 - 1e-6), (t > s.thr).float())
        tv = (p[:, 1:] - p[:, :-1]).abs().mean() + (p[:, :, 1:] - p[:, :, :-1]).abs().mean() if s.g > 0 else 0.0
        return s.a * wmse + s.b * bce + s.g * tv
    def forward(s, p, t):                                               # p: (B,H,W)|(B,P,H,W) ; t cùng dạng
        if p.dim() == 3: return s.step(p, t)
        return sum(s.sw[k] * s.step(p[:, k], t[:, k]) for k in range(p.shape[1]))

def get_batch(split, idx, P):
    Zs = Z[split]; x = Zs[idx[:, None] + AR6].unsqueeze(2)                      # (B,T,1,H,W)
    y = Zs[idx[:, None] + CFG['SEQ_LEN'] + torch.arange(P, device=DEVICE)]      # (B,P,H,W)
    return x, (y[:, 0] if P == 1 else y)

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

def tf_ratio(ep, total, start=0.9, end=0.1):
    return end if ep >= total else start - (start - end) * ep / total

@torch.no_grad()
def val_loss(model, kind, crit, P):
    model.eval(); n = len(Z['val']) - CFG['SEQ_LEN'] - P + 1; tot, cnt = 0.0, 0
    for s in range(0, n, CFG['EVAL_BATCH']):
        idx = torch.arange(s, min(s + CFG['EVAL_BATCH'], n), device=DEVICE); x, y = get_batch('val', idx, P)
        out = model(x); tot += float(crit(out.float(), y)) * len(idx); cnt += len(idx)
    return tot / cnt

def train_run(name, seed, batch=None):
    mk, kind, lkey = REG[name]; P = 3 if kind == 's2s' else 1
    epochs = CFG['S2S_EPOCHS'] if kind == 's2s' else CFG['EPOCHS']; B = batch or CFG['BATCH']
    set_seed(seed); model = mk().to(DEVICE)
    n_tr = len(Z['train']) - CFG['SEQ_LEN'] - P + 1; steps = n_tr // B
    hp = CFG.get('HP', {}).get(name, {})
    opt = torch.optim.AdamW(model.parameters(), lr=hp.get('LR', CFG['LR']), weight_decay=CFG['WD'])
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=hp.get('MAX_LR', CFG['MAX_LR']), total_steps=epochs * steps, pct_start=0.15)
    crit = RainLoss(**CFG[lkey], P=P); ck_last = f'{OUT}/ckpt/{name}_s{seed}_last.pt'; ck_best = f'{OUT}/ckpt/{name}_s{seed}_best.pt'
    hist, best, bad, ep0 = [], 1e9, 0, 0
    if os.path.exists(ck_last):
        ck = torch.load(ck_last, map_location=DEVICE); model.load_state_dict(ck['model']); opt.load_state_dict(ck['opt'])
        sched.load_state_dict(ck['sched']); hist, best, bad, ep0 = ck['hist'], ck['best'], ck['bad'], ck['epoch']
        print(f'   ↻ resume {name} s{seed} từ epoch {ep0}')
    g = torch.Generator(device=DEVICE); g.manual_seed(seed * 1000 + ep0)
    use_amp = CFG['USE_AMP'] and DEVICE.type == 'cuda'; t_start = time.time()
    for ep in range(ep0, epochs):
        model.train(); perm = torch.randperm(n_tr, device=DEVICE, generator=g); tl = 0.0; tf = tf_ratio(ep, min(40, epochs))
        for it in range(steps):
            x, y = get_batch('train', perm[it * B:(it + 1) * B], P)
            with torch.autocast(DEVICE.type, dtype=torch.bfloat16, enabled=use_amp):
                out = model(x, y, tf) if kind == 's2s' else model(x)
            loss = crit(out.float(), y)
            aux = getattr(model, 'aux', None)                            # PredRNN++: thêm loss trung bình trên các khung đầu vào
            if aux is not None: loss = loss + sum(crit.step(aux[0][:, k].float(), aux[1][:, k]) for k in range(aux[0].shape[1])) / aux[0].shape[1]
            opt.zero_grad(set_to_none=True); loss.backward(); nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step(); sched.step()
            tl += float(loss)
        vl = val_loss(model, kind, crit, P); hist.append(dict(epoch=ep + 1, train=tl / steps, val=vl, lr=sched.get_last_lr()[0], tf=tf))
        if vl < best: best, bad = vl, 0; torch.save(model.state_dict(), ck_best)
        else: bad += 1
        torch.save(dict(model=model.state_dict(), opt=opt.state_dict(), sched=sched.state_dict(), hist=hist, best=best, bad=bad, epoch=ep + 1), ck_last)
        print(f'   {name} s{seed} ep {ep + 1:>2}/{epochs}  train {tl / steps:.5f}  val {vl:.5f}  best {best:.5f}  tf {tf:.2f}  ({time.time() - t_start:.0f}s)', flush=True)
        if bad >= CFG['PATIENCE']: print('   early stop'); break
    model.load_state_dict(torch.load(ck_best, map_location=DEVICE))
    best_ep = int(np.argmin([h['val'] for h in hist])) + 1
    return model, dict(epochs_run=len(hist), best_epoch=best_ep, best_val=best, history=hist, batch=B, train_seconds=time.time() - t_start)
print('training utilities ready')

# ── 8. DỰ BÁO + ĐÁNH GIÁ (raw / cùng bias / best-CSI) ────────────────
@torch.no_grad()
def predict(model, kind, split):
    model.eval(); n = NS[split]; nh = 3 if (kind == 's2s' or CFG['ROLLOUT']) else 1
    out = torch.empty((n, nh, H, W), dtype=torch.float16, device=DEVICE)
    for s in range(0, n, CFG['EVAL_BATCH']):
        idx = torch.arange(s, min(s + CFG['EVAL_BATCH'], n), device=DEVICE); x = Z[split][idx[:, None] + AR6].unsqueeze(2)
        if kind == 's2s': out[idx] = model(x).half()
        else:
            for k in range(nh):
                p = model(x); out[idx, k] = p.half(); x = torch.cat([x[:, 1:], p[:, None, None]], 1)
    return out

def count_gt(P, tau): return int((P > tau).sum())

def find_tau_bias1(Pv, Ov):                                             # chọn τ để số điểm dự báo mưa = số điểm mưa thực (bias = 1) trên VAL
    K = int((Ov > EVENT_THR).sum()); lo, hi = 1e-4, float(Pv.max())
    for _ in range(40):
        mid = (lo * hi) ** 0.5
        if count_gt(Pv, mid) > K: lo = mid
        else: hi = mid
    return (lo * hi) ** 0.5

def find_tau_csi(Pv, Ov):                                               # τ tối đa hóa CSI trên VAL (ngưỡng tune kiểu NB3, nhưng không nhìn test)
    Ob = Ov > EVENT_THR; best = (-1, None)
    for tau in np.geomspace(0.05, max(6.0, float(Pv.max())), 50):
        pb = Pv > tau; h = int((pb & Ob).sum()); m = int((~pb & Ob).sum()); f = int((pb & ~Ob).sum()); c = h / max(h + m + f, 1)
        if c > best[0]: best = (c, float(tau))
    return best[1]

@torch.no_grad()
def evaluate_model(name, seed, model, kind, train_info, save_preds):
    Pv, Pt = predict(model, kind, 'val'), predict(model, kind, 'test'); nh = Pt.shape[1]
    out = dict(name=name, seed=seed, params=NPARAM[name], train=train_info, h={}); cnt = {}
    for h in range(1, nh + 1):
        Pvh = den(Pv[:, h - 1].float()); Ovh = obs_of('val', h, 0, NS['val'])
        taus = dict(raw=EVENT_THR, bias1=find_tau_bias1(Pvh, Ovh), bestcsi=find_tau_csi(Pvh, Ovh)); del Pvh
        rec = dict(tau=taus, thr={})
        for mode, tau in taus.items():
            a = Acc(); sc = EVENT_THR / tau
            for i0 in range(0, NS['test'], 128):
                i1 = min(i0 + 128, NS['test']); a.add(den(Pt[i0:i1, h - 1].float()) * sc, obs_of('test', h, i0, i1))
            rec[mode] = a.res(with_rmse=(mode == 'raw')); cnt[f'h{h}_{mode}'] = a.counts()
            if mode == 'raw': rec['thr'] = {str(t): a.res(t) for t in THR}
        out['h'][str(h)] = rec
    save_json(f'{OUT}/results/{name}_s{seed}.json', out); np.savez_compressed(f'{OUT}/results/{name}_s{seed}_counts.npz', **cnt)
    if save_preds: np.save(f'{OUT}/preds/{name}_s{seed}_test.npy', Pt.cpu().numpy())
    return out

def run_one(name, seed):
    if os.path.exists(f'{OUT}/results/{name}_s{seed}.json'): print(f'✓ bỏ qua {name} s{seed} (đã có kết quả)'); return
    _, kind, _ = REG[name]
    for B in (CFG['BATCH'], CFG['BATCH'] // 2, CFG['BATCH'] // 4):
        try:
            model, info = train_run(name, seed, B); break
        except torch.cuda.OutOfMemoryError:
            print(f'   OOM với batch {B} → thử nhỏ hơn'); torch.cuda.empty_cache()
    r = evaluate_model(name, seed, model, kind, info, save_preds=(seed == CFG['SAVE_PREDS_SEED']))
    h1 = r['h']['1']; print(f'★ {name} s{seed}  +30: raw CSI {h1["raw"]["CSI"]:.4f} POD {h1["raw"]["POD"]:.4f} FAR {h1["raw"]["FAR"]:.4f} Bias {h1["raw"]["Bias"]:.2f} | bias-matched CSI {h1["bias1"]["CSI"]:.4f}')
    del model; torch.cuda.empty_cache() if DEVICE.type == 'cuda' else None
print('evaluation ready')

Mounted at /content/drive
GPU: NVIDIA L4 | VRAM 23.7 GB
torch 2.14.1+cu130 | cv2 4.14.0
OUT_DIR: /content/drive/MyDrive/rainfall_p0 
{
 "T": 27869,
 "H": 150,
 "W": 80,
 "n_train": 19508,
 "n_val": 4180,
 "n_test": 4181,
 "n_eval_samples": {
  "val": 4172,
  "test": 4173
 },
 "log_max_train": 4.684628009796143,
 "raw_max_all": 107.2699966430664,
 "raw_max_train": 107.2699966430664,
 "frac_zero": 0.8412021003026062,
 "frac_gt_0p1": 0.13049012343464064,
 "frac_gt_10": 0.0022605882282583996,
 "test_pixels_above_train_max": 0.0,
 "frac_rain_gt_0p5": {
  "train": 0.08179973600574124,
  "val": 0.01982422248803828,
  "test": 0.11923742326397194
 }
}
27869 file .nc4, 27869 có timestamp trong tên
Khoảng thời gian: 2022-01-01 00:00:00 → 2023-08-04 14:00:00 | số bước kỳ vọng nếu liên tục: 27869
Số khoảng trống (bước ≠ 30 phút): 0
Tập test: 2023-05-09 12:00:00 → 2023-08-04 14:00:00
metrics ready
persistence (5s) {'1': {'RMSE': 0.8039, 'CSI': 0.6554, 'POD': 0.7918, 'FAR': 0.2081, 'Bias': 0.9998}, '

In [2]:
# ── V4. THỰC NGHIỆM BỔ SUNG: ngưỡng nhãn của BCE (RainS2S) ───────────────────────────────────────────
# Mặc định trong bài: thr = 0.05 (chuẩn hóa) ≈ 0.26 mm/h. Thử thr ≈ 0.5 mm/h (trùng ngưỡng đánh giá) và ≈ 1 mm/h.
# Kết quả của thr = 0.05 lấy từ lần chạy trước (results/seq2seq_unet_s{seed}.json), không huấn luyện lại.
thr_of = lambda mm: float(np.log1p(mm) / LOG_MAX)
VAR = {'seq2seq_unet_bce05': 0.5, 'seq2seq_unet_bce10': 1.0}          # tên → ngưỡng nhãn BCE (mm/h)
for nm, mm in VAR.items():
    CFG['LOSS_' + nm] = dict(CFG['LOSS'], thr=thr_of(mm))
    REG[nm] = (lambda: ConvLSTMUNet(pred_len=3), 's2s', 'LOSS_' + nm)
    NPARAM[nm] = NPARAM['seq2seq_unet']
    print(nm, 'BCE thr chuẩn hóa =', round(CFG['LOSS_' + nm]['thr'], 4), '≈', mm, 'mm/h')
for nm in VAR:
    for seed in CFG['SEEDS']:
        run_one(nm, seed)

# ── Tổng hợp: so 3 ngưỡng BCE (0.26 / 0.5 / 1 mm/h) ──
def load(nm, seed):
    return json.load(open(f'{OUT}/results/{nm}_s{seed}.json'))
rows = []
for label, nm, mm in [('0,26 (mặc định)', 'seq2seq_unet', 0.26), ('0,5', 'seq2seq_unet_bce05', 0.5), ('1,0', 'seq2seq_unet_bce10', 1.0)]:
    rs = [load(nm, s) for s in CFG['SEEDS']]
    for h in (1, 2, 3):
        g = lambda mode, k: np.array([r['h'][str(h)][mode][k] for r in rs])
        rows.append(dict(bce_thr_mmh=label, lead=f'+{30*h}', n_seeds=len(rs),
                         CSI_raw=g('raw', 'CSI').mean(), CSI_raw_sd=g('raw', 'CSI').std(), POD_raw=g('raw', 'POD').mean(),
                         FAR_raw=g('raw', 'FAR').mean(), Bias_raw=g('raw', 'Bias').mean(), RMSE=g('raw', 'RMSE').mean(),
                         CSI_cal=g('bias1', 'CSI').mean(), CSI_cal_sd=g('bias1', 'CSI').std(), Bias_cal=g('bias1', 'Bias').mean()))
df = pd.DataFrame(rows); df.to_csv(f'{OUT}/tables/T7_bce_threshold_ablation.csv', index=False)
pd.set_option('display.width', 220, 'display.max_columns', 30)
print(df.round(4).to_string(index=False))
if IN_COLAB:
    shutil.make_archive('/content/T7_bce_ablation', 'zip', f'{OUT}/tables')
    from google.colab import files; files.download('/content/T7_bce_ablation.zip')

seq2seq_unet_bce05 BCE thr chuẩn hóa = 0.0866 ≈ 0.5 mm/h
seq2seq_unet_bce10 BCE thr chuẩn hóa = 0.148 ≈ 1.0 mm/h
   seq2seq_unet_bce05 s0 ep  1/40  train 0.21693  val 0.14223  best 0.14223  tf 0.90  (226s)
   seq2seq_unet_bce05 s0 ep  2/40  train 0.10939  val 0.03841  best 0.03841  tf 0.88  (430s)
   seq2seq_unet_bce05 s0 ep  3/40  train 0.05742  val 0.02145  best 0.02145  tf 0.86  (634s)
   seq2seq_unet_bce05 s0 ep  4/40  train 0.04772  val 0.03018  best 0.02145  tf 0.84  (837s)
   seq2seq_unet_bce05 s0 ep  5/40  train 0.04398  val 0.03488  best 0.02145  tf 0.82  (1041s)
   seq2seq_unet_bce05 s0 ep  6/40  train 0.04303  val 0.01569  best 0.01569  tf 0.80  (1245s)
   seq2seq_unet_bce05 s0 ep  7/40  train 0.04262  val 0.01567  best 0.01567  tf 0.78  (1448s)
   seq2seq_unet_bce05 s0 ep  8/40  train 0.04230  val 0.01768  best 0.01567  tf 0.76  (1652s)
   seq2seq_unet_bce05 s0 ep  9/40  train 0.04212  val 0.02135  best 0.01567  tf 0.74  (1856s)
   seq2seq_unet_bce05 s0 ep 10/40  train 0.04

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>